# App-28 — Learning to branch : auditer la généralisation avant d'annoncer un gain

[← Applications](../README.md) | [↑ Search](../../README.md) | [<< App-25 Enchères WDP/VCG](App-25-CombinatorialAuctions-WDP-VCG.ipynb)

> **Durée estimée : 60 minutes**

## Hommage et périmètre

Ce notebook distille le geste du groupe **G4 — Simon Naulet et Matis Codjia** : apprendre une politique de choix de variable pour un solveur CSP. Source : [PR PrCon #46](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/pull/46), fusion `9f91222f`. Le notebook étudiant a correctement construit un mini-solveur, AC-3, des features de branchement et une imitation de `dom/wdeg` par XGBoost.

La reproduction fraîche CoursIA ne copie aucune cellule, fonction, donnée, figure ou prose étudiante. Elle reconstruit le dispositif avec un protocole qui empêche la fuite par instance et pose une question plus stricte : **la politique apprise généralise-t-elle à de nouvelles instances et à une famille CSP entièrement absente du train, une fois son coût d'inférence compté ?**

### Objectifs

1. Distinguer split par candidats, split par instances et famille tenue à l'écart.
2. Mesurer séparément fidélité à l'oracle, nœuds, temps mur et part d'inférence.
3. Sélectionner la baseline sur le train uniquement.
4. Accepter un résultat négatif comme résultat scientifique.

### Prérequis

- [CSP-6 Hybridation](../../Part2-CSP/CSP-6-Hybridization.ipynb)
- [MGS-16 Sélection d'algorithmes](../../Part4-Metaheuristics/MGS-16-AlgorithmSelection.ipynb)
- Python 3.10+ ; `numpy`, `pandas`, `scikit-learn`, `matplotlib`


## 1. Pourquoi le split naïf est trompeur

À un nœud de recherche, chaque variable candidate produit une ligne de features. Un split aléatoire de ces **lignes** place facilement des candidats du même nœud — et des nœuds de la même instance — dans le train et le test. L'accuracy binaire est en outre dominée par les nombreux candidats non choisis.

Nous évaluons donc le **top-1 par nœud**, puis la performance intégrée du solveur. Les identifiants d'instance sont disjoints entre train et test. Trois validations supplémentaires tiennent successivement reines, coloration et carrés latins entièrement à l'écart.


In [1]:
from __future__ import annotations

import json
import math
import random
import time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier


print('Imports chargés : numpy, pandas et scikit-learn')

Imports chargés : numpy, pandas et scikit-learn


## 2. Exemple guide — un solveur et un validateur indépendants

Le solveur utilise des domaines finis, des contraintes binaires, AC-3 et un backtracking chronologique. La politique de branchement est pluggable. Le validateur final ne fait confiance ni au statut ni au chemin de recherche : il recalcule domaines et contraintes sur l'affectation retournée.


In [2]:

Relation = Callable[[int, int], bool]
FEATURES = [
    "domain_size",
    "domain_ratio",
    "degree",
    "degree_ratio",
    "weighted_degree",
    "activity",
    "dom_wdeg",
    "progress",
    "neighbor_domain_min",
    "neighbor_domain_mean",
    "neighbor_domain_max",
]
BASELINES = ("mrv", "max_degree", "dom_wdeg", "activity")


@dataclass
class CSP:
    family: str
    instance_id: str
    domains: dict[str, set[int]] = field(default_factory=dict)
    neighbors: dict[str, set[str]] = field(default_factory=dict)
    relations: dict[tuple[str, str], list[Relation]] = field(default_factory=dict)

    @property
    def variables(self) -> list[str]:
        return list(self.domains)

    def add_variable(self, name: str, domain: range | list[int] | set[int]) -> None:
        self.domains[name] = set(domain)
        self.neighbors[name] = set()

    def add_constraint(self, left: str, right: str, relation: Relation) -> None:
        self.neighbors[left].add(right)
        self.neighbors[right].add(left)
        self.relations.setdefault((left, right), []).append(relation)
        self.relations.setdefault((right, left), []).append(lambda b, a, rel=relation: rel(a, b))

    def add_all_different(self, variables: list[str]) -> None:
        for i, left in enumerate(variables):
            for right in variables[i + 1 :]:
                self.add_constraint(left, right, lambda a, b: a != b)


def revise(csp: CSP, domains: dict[str, set[int]], left: str, right: str) -> bool:
    relations = csp.relations.get((left, right), [])
    remove = {
        value
        for value in domains[left]
        if not any(all(rel(value, other) for rel in relations) for other in domains[right])
    }
    if remove:
        domains[left] -= remove
        return True
    return False


def ac3(
    csp: CSP,
    domains: dict[str, set[int]],
    arcs: list[tuple[str, str]] | None = None,
) -> tuple[bool, set[str]]:
    queue = deque(arcs if arcs is not None else csp.relations)
    changed: set[str] = set()
    while queue:
        left, right = queue.popleft()
        if revise(csp, domains, left, right):
            changed.add(left)
            if not domains[left]:
                return False, changed
            queue.extend((neighbor, left) for neighbor in csp.neighbors[left] if neighbor != right)
    return True, changed


def features_for(
    csp: CSP,
    domains: dict[str, set[int]],
    assignment: dict[str, int],
    weights: dict[tuple[str, str], int],
    activity: dict[str, int],
    variable: str,
) -> dict[str, float]:
    remaining = [neighbor for neighbor in csp.neighbors[variable] if neighbor not in assignment]
    weighted_degree = sum(weights.get(tuple(sorted((variable, neighbor))), 1) for neighbor in remaining)
    neighbor_sizes = [len(domains[neighbor]) for neighbor in remaining]
    n_variables = len(csp.variables)
    initial_size = len(csp.domains[variable])
    return {
        "domain_size": float(len(domains[variable])),
        "domain_ratio": len(domains[variable]) / max(initial_size, 1),
        "degree": float(len(remaining)),
        "degree_ratio": len(remaining) / max(n_variables - 1, 1),
        "weighted_degree": float(weighted_degree),
        "activity": float(activity.get(variable, 0)),
        "dom_wdeg": len(domains[variable]) / max(weighted_degree, 1),
        "progress": len(assignment) / max(n_variables, 1),
        "neighbor_domain_min": float(min(neighbor_sizes, default=0)),
        "neighbor_domain_mean": float(np.mean(neighbor_sizes) if neighbor_sizes else 0),
        "neighbor_domain_max": float(max(neighbor_sizes, default=0)),
    }


class Solver:
    def __init__(
        self,
        csp: CSP,
        heuristic: str | Callable[..., str],
        collect_trace: bool = False,
        node_limit: int = 20_000,
    ) -> None:
        self.csp = csp
        self.heuristic = heuristic
        self.collect_trace = collect_trace
        self.node_limit = node_limit
        self.nodes = 0
        self.inference_seconds = 0.0
        self.trace: list[dict[str, object]] = []
        self.weights: dict[tuple[str, str], int] = {}
        self.activity: dict[str, int] = {}

    def select(self, domains: dict[str, set[int]], assignment: dict[str, int]) -> str:
        candidates = [variable for variable in self.csp.variables if variable not in assignment]
        rows = [features_for(self.csp, domains, assignment, self.weights, self.activity, var) for var in candidates]
        if callable(self.heuristic):
            started = time.perf_counter()
            chosen = self.heuristic(candidates, rows)
            self.inference_seconds += time.perf_counter() - started
        elif self.heuristic == "mrv":
            chosen = min(candidates, key=lambda var: (len(domains[var]), var))
        elif self.heuristic == "max_degree":
            chosen = min(
                candidates,
                key=lambda var: (-sum(n not in assignment for n in self.csp.neighbors[var]), var),
            )
        elif self.heuristic == "dom_wdeg":
            chosen = min(
                candidates,
                key=lambda var: (
                    features_for(self.csp, domains, assignment, self.weights, self.activity, var)["dom_wdeg"],
                    var,
                ),
            )
        elif self.heuristic == "activity":
            chosen = min(candidates, key=lambda var: (-self.activity.get(var, 0), var))
        else:
            raise ValueError(f"Unknown heuristic: {self.heuristic}")

        if self.collect_trace and len(candidates) > 1:
            node_id = f"{self.csp.instance_id}:{self.nodes}"
            for candidate, row in zip(candidates, rows):
                self.trace.append(
                    {
                        "family": self.csp.family,
                        "instance_id": self.csp.instance_id,
                        "node_id": node_id,
                        "candidate": candidate,
                        **row,
                        "label": int(candidate == chosen),
                    }
                )
        return chosen

    def solve(self) -> dict[str, int] | None:
        domains = {variable: set(values) for variable, values in self.csp.domains.items()}
        consistent, _ = ac3(self.csp, domains)
        if not consistent:
            return None
        return self._search({}, domains)

    def _search(self, assignment: dict[str, int], domains: dict[str, set[int]]) -> dict[str, int] | None:
        if len(assignment) == len(self.csp.variables):
            return dict(assignment)
        if self.nodes >= self.node_limit:
            return None
        self.nodes += 1
        variable = self.select(domains, assignment)
        for value in sorted(domains[variable]):
            next_domains = {name: set(values) for name, values in domains.items()}
            next_assignment = dict(assignment)
            next_assignment[variable] = value
            next_domains[variable] = {value}
            consistent, changed = ac3(
                self.csp,
                next_domains,
                [(neighbor, variable) for neighbor in self.csp.neighbors[variable]],
            )
            for changed_variable in changed:
                self.activity[changed_variable] = self.activity.get(changed_variable, 0) + 1
            if consistent:
                result = self._search(next_assignment, next_domains)
                if result is not None:
                    return result
            for neighbor in self.csp.neighbors[variable]:
                key = tuple(sorted((variable, neighbor)))
                self.weights[key] = self.weights.get(key, 1) + 1
        return None



print('Noyau CSP prêt : propagation AC-3, recherche et validation indépendante')

Noyau CSP prêt : propagation AC-3, recherche et validation indépendante


### Interprétation

La séparation entre résolution et validation protège l'expérience contre un faux gain obtenu en retournant une affectation incomplète ou invalide. `FEASIBLE` signifie ici « affectation complète validée », jamais « meilleure politique ».

## 3. Instances déterministes et distinctes

Les 36 CSP sont générés sans réseau : douze tailles de N-reines, douze colorations plantées et douze carrés latins partiellement révélés. Chaque `instance_id` appartient à un seul côté du split.


In [3]:

def make_queens(index: int) -> CSP:
    n = 8 + index
    csp = CSP("queens", f"queens-{n}")
    for row in range(n):
        csp.add_variable(f"q{row}", range(n))
    for left in range(n):
        for right in range(left + 1, n):
            distance = right - left
            csp.add_constraint(
                f"q{left}",
                f"q{right}",
                lambda a, b, d=distance: a != b and abs(a - b) != d,
            )
    return csp


def make_coloring(index: int) -> CSP:
    seed = 10_000 + index
    rng = random.Random(seed)
    n = 16 + index
    colors = 3 + index % 2
    planted = [rng.randrange(colors) for _ in range(n)]
    density = 0.28 + 0.02 * (index % 5)
    edges = [
        (left, right)
        for left in range(n)
        for right in range(left + 1, n)
        if planted[left] != planted[right] and rng.random() < density
    ]
    csp = CSP("coloring", f"coloring-{index:02d}")
    for node in range(n):
        csp.add_variable(f"v{node}", range(colors))
    for left, right in edges:
        csp.add_constraint(f"v{left}", f"v{right}", lambda a, b: a != b)
    return csp


def make_latin(index: int) -> CSP:
    seed = 20_000 + index
    rng = random.Random(seed)
    n = 4 + index % 4
    row_perm = list(range(n))
    col_perm = list(range(n))
    value_perm = list(range(n))
    rng.shuffle(row_perm)
    rng.shuffle(col_perm)
    rng.shuffle(value_perm)
    solution = {
        (row, col): value_perm[(row_perm[row] + col_perm[col]) % n]
        for row in range(n)
        for col in range(n)
    }
    reveal_rate = 0.18 + 0.03 * (index % 4)
    csp = CSP("latin", f"latin-{index:02d}")
    for row in range(n):
        for col in range(n):
            domain = [solution[(row, col)]] if rng.random() < reveal_rate else list(range(n))
            csp.add_variable(f"x{row}_{col}", domain)
    for row in range(n):
        csp.add_all_different([f"x{row}_{col}" for col in range(n)])
    for col in range(n):
        csp.add_all_different([f"x{row}_{col}" for row in range(n)])
    return csp


def build_instances(count_per_family: int = 12) -> list[CSP]:
    makers = (make_queens, make_coloring, make_latin)
    return [maker(index) for maker in makers for index in range(count_per_family)]


def validate_solution(csp: CSP, solution: dict[str, int] | None) -> bool:
    if solution is None or set(solution) != set(csp.variables):
        return False
    if any(solution[var] not in csp.domains[var] for var in csp.variables):
        return False
    return all(
        all(relation(solution[left], solution[right]) for relation in relations)
        for (left, right), relations in csp.relations.items()
    )


def run_solver(csp: CSP, heuristic: str | Callable[..., str], collect_trace: bool = False) -> tuple[dict, list[dict]]:
    solver = Solver(csp, heuristic, collect_trace=collect_trace)
    started = time.perf_counter()
    solution = solver.solve()
    elapsed = time.perf_counter() - started
    result = {
        "family": csp.family,
        "instance_id": csp.instance_id,
        "heuristic": heuristic if isinstance(heuristic, str) else "learned_policy",
        "solved": solution is not None,
        "valid": validate_solution(csp, solution),
        "nodes": solver.nodes,
        "seconds": elapsed,
        "inference_seconds": solver.inference_seconds,
    }
    return result, solver.trace



print('Générateurs déterministes prêts : reines, coloration plantée et carrés latins')

Générateurs déterministes prêts : reines, coloration plantée et carrés latins


### Interprétation

Les trois familles ne prétendent pas représenter toute la programmation par contraintes. Elles offrent un banc synthétique contrôlé pour tester la fuite et le transfert. Les conclusions restent bornées à la recherche de faisabilité sur CSP binaires de petite taille.

## 4. Politique apprise et baseline sélectionnée sans regarder le test

La cible reste l'imitation pointwise de `dom/wdeg`, mais l'évaluation est structurée par nœud. Le classifieur `HistGradientBoostingClassifier` évite une dépendance XGBoost supplémentaire. Pour chaque split, la baseline déployable est choisie par médiane des nœuds **sur le train seulement** parmi MRV, max-degree, dom/wdeg et activity.


In [4]:

def train_policy(
    trace: pd.DataFrame,
    instance_ids: set[str],
    random_state: int,
) -> HistGradientBoostingClassifier:
    rows = trace[trace["instance_id"].isin(instance_ids)]
    model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=160,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.1,
        random_state=random_state,
    )
    model.fit(rows[FEATURES], rows["label"])
    return model


def learned_heuristic(model: HistGradientBoostingClassifier) -> Callable[..., str]:
    def choose(candidates: list[str], rows: list[dict[str, float]]) -> str:
        frame = pd.DataFrame(rows, columns=FEATURES)
        scores = model.predict_proba(frame)[:, 1]
        best = max(range(len(candidates)), key=lambda index: (scores[index], -index))
        return candidates[best]

    return choose


def top1_accuracy(model: HistGradientBoostingClassifier, trace: pd.DataFrame, instance_ids: set[str]) -> float:
    rows = trace[trace["instance_id"].isin(instance_ids)].copy()
    rows["score"] = model.predict_proba(rows[FEATURES])[:, 1]
    predicted = rows.loc[rows.groupby("node_id")["score"].idxmax(), ["node_id", "label"]]
    return float(predicted["label"].mean())


def choose_train_baseline(baselines: pd.DataFrame, train_ids: set[str]) -> str:
    train = baselines[baselines["instance_id"].isin(train_ids)]
    medians = train.groupby("heuristic")["nodes"].median().sort_values(kind="stable")
    return str(medians.index[0])


def summarize_evaluation(frame: pd.DataFrame, split_name: str, baseline: str) -> dict:
    learned = frame[frame["heuristic"] == "learned_policy"].set_index("instance_id")
    selected = frame[frame["heuristic"] == baseline].set_index("instance_id")
    merged = learned.join(selected, lsuffix="_ml", rsuffix="_baseline")
    node_ratio = merged["nodes_ml"] / merged["nodes_baseline"].clip(lower=1)
    time_ratio = merged["seconds_ml"] / merged["seconds_baseline"].clip(lower=1e-9)
    strict_node_wins = int((merged["nodes_ml"] < merged["nodes_baseline"]).sum())
    strict_time_wins = int((merged["seconds_ml"] < merged["seconds_baseline"]).sum())
    return {
        "split": split_name,
        "selected_baseline": baseline,
        "instances": len(merged),
        "all_solved": bool(merged["solved_ml"].all() and merged["solved_baseline"].all()),
        "median_node_ratio": float(node_ratio.median()),
        "geomean_node_ratio": float(math.exp(np.log(node_ratio).mean())),
        "median_time_ratio": float(time_ratio.median()),
        "strict_node_wins": strict_node_wins,
        "strict_time_wins": strict_time_wins,
        "median_inference_share": float((merged["inference_seconds_ml"] / merged["seconds_ml"]).median()),
    }



print('Politique apprise et métriques de comparaison prêtes')

Politique apprise et métriques de comparaison prêtes


### Interprétation

Le modèle ne peut, par construction, apprendre à dépasser systématiquement son oracle : il apprend surtout une approximation coûteuse de `dom/wdeg`. C'est précisément l'hypothèse testée, pas un gain supposé.

## 5. Protocole complet

Chaque split est répété avec cinq graines du modèle. Les mêmes CSP sont présentés à la politique apprise et à la baseline sélectionnée. Nous rapportons le ratio de nœuds, le ratio de temps, les victoires strictes et la part du temps consommée par l'inférence.


In [5]:

def main(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    instances = build_instances()

    baseline_rows: list[dict] = []
    trace_rows: list[dict] = []
    for index, csp in enumerate(instances, start=1):
        for heuristic in BASELINES:
            result, trace = run_solver(csp, heuristic, collect_trace=heuristic == "dom_wdeg")
            baseline_rows.append(result)
            if heuristic == "dom_wdeg":
                trace_rows.extend(trace)
        print(f"baseline {index:02d}/{len(instances)} {csp.instance_id}", flush=True)

    baselines = pd.DataFrame(baseline_rows)
    trace = pd.DataFrame(trace_rows)
    baselines.to_csv(output_dir / "baseline_runs.csv", index=False)
    trace.to_csv(output_dir / "oracle_trace.csv", index=False)

    splits: list[tuple[str, set[str], set[str]]] = []
    grouped_train = {csp.instance_id for csp in instances if int(csp.instance_id.rsplit("-", 1)[1]) < 8 or csp.family == "queens" and int(csp.instance_id.rsplit("-", 1)[1]) < 16}
    all_ids = {csp.instance_id for csp in instances}
    splits.append(("grouped_instance_split", grouped_train, all_ids - grouped_train))
    for held_out in ("queens", "coloring", "latin"):
        test_ids = {csp.instance_id for csp in instances if csp.family == held_out}
        splits.append((f"leave_{held_out}_out", all_ids - test_ids, test_ids))

    summaries: list[dict] = []
    evaluation_rows: list[dict] = []
    model_seeds = [11, 23, 42, 71, 101]
    for split_name, train_ids, test_ids in splits:
        baseline = choose_train_baseline(baselines, train_ids)
        for model_seed in model_seeds:
            model = train_policy(trace, train_ids, model_seed)
            policy = learned_heuristic(model)
            split_rows: list[dict] = []
            for csp in instances:
                if csp.instance_id not in test_ids:
                    continue
                result, _ = run_solver(csp, policy)
                result["split"] = split_name
                result["model_seed"] = model_seed
                split_rows.append(result)
                evaluation_rows.append(result)
                selected = baselines[
                    (baselines["instance_id"] == csp.instance_id)
                    & (baselines["heuristic"] == baseline)
                ].iloc[0].to_dict()
                selected["split"] = split_name
                selected["model_seed"] = model_seed
                split_rows.append(selected)
                evaluation_rows.append(selected)
            split_frame = pd.DataFrame(split_rows)
            summary = summarize_evaluation(split_frame, split_name, baseline)
            summary["model_seed"] = model_seed
            summary["top1_node_accuracy"] = top1_accuracy(model, trace, test_ids)
            summary["train_instances"] = len(train_ids)
            summary["test_instances"] = len(test_ids)
            summary["all_valid"] = bool(split_frame["valid"].all())
            summaries.append(summary)
            print(json.dumps(summary, ensure_ascii=False), flush=True)

    pd.DataFrame(evaluation_rows).to_csv(output_dir / "evaluation_runs.csv", index=False)
    report = {
        "protocol": {
            "families": ["queens", "coloring", "latin"],
            "instances_per_family": 12,
            "labels": "dom/wdeg imitation at candidate level",
            "classifier": "sklearn HistGradientBoostingClassifier",
            "model_seeds": model_seeds,
            "selection_rule": "baseline selected by median train nodes only",
            "validation": "all returned assignments checked independently against domains and binary constraints",
            "splits": [name for name, _, _ in splits],
            "metrics": ["node top-1", "nodes", "wall time", "inference share"],
            "scientific_scope": "small synthetic binary CSPs; feasibility search only",
        },
        "dataset": {
            "trace_rows": len(trace),
            "decision_nodes": int(trace["node_id"].nunique()),
            "instances": len(instances),
        },
        "results": summaries,
    }
    (output_dir / "maturation_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )



print('Protocole expérimental prêt : split groupé et leave-one-family-out')

Protocole expérimental prêt : split groupé et leave-one-family-out


## 6. Exécution fraîche

Les CSV et le rapport JSON sont régénérés sous `data/app28-learning-to-branch-audit/`. Les temps absolus restent dépendants de la machine ; les cellules suivantes interprètent surtout les ratios mesurés dans ce run.


In [6]:
from pathlib import Path

OUTPUT_DIR = Path("data/app28-learning-to-branch-audit")
main(OUTPUT_DIR)
print(f"Artefacts frais écrits dans {OUTPUT_DIR.as_posix()}")


baseline 01/36 queens-8


baseline 02/36 queens-9


baseline 03/36 queens-10


baseline 04/36 queens-11


baseline 05/36 queens-12


...

{"split": "leave_latin_out", "selected_baseline": "mrv", "instances": 12, "all_solved": true, "median_node_ratio": 1.0, "geomean_node_ratio": 1.0184220950158045, "median_time_ratio": 9.055735717636894, "strict_node_wins": 0, "strict_time_wins": 0, "median_inference_share": 0.8529476093437629, "model_seed": 23, "top1_node_accuracy": 0.546448087431694, "train_instances": 24, "test_instances": 12, "all_valid": true}


{"split": "leave_latin_out", "selected_baseline": "mrv", "instances": 12, "all_solved": true, "median_node_ratio": 1.0, "geomean_node_ratio": 1.0184220950158045, "median_time_ratio": 9.300364390125992, "strict_node_wins": 0, "strict_time_wins": 0, "median_inference_share": 0.847386129802411, "model_seed": 42, "top1_node_accuracy": 0.546448087431694, "train_instances": 24, "test_instances": 12, "all_valid": true}


{"split": "leave_latin_out", "selected_baseline": "mrv", "instances": 12, "all_solved": true, "median_node_ratio": 1.0, "geomean_node_ratio": 1.0184220950158045, "median_time_ratio": 9.683559298336608, "strict_node_wins": 0, "strict_time_wins": 0, "median_inference_share": 0.8530324183614684, "model_seed": 71, "top1_node_accuracy": 0.546448087431694, "train_instances": 24, "test_instances": 12, "all_valid": true}


{"split": "leave_latin_out", "selected_baseline": "mrv", "instances": 12, "all_solved": true, "median_node_ratio": 1.0, "geomean_node_ratio": 1.0184220950158045, "median_time_ratio": 11.538061789488488, "strict_node_wins": 0, "strict_time_wins": 0, "median_inference_share": 0.8551428036154662, "model_seed": 101, "top1_node_accuracy": 0.546448087431694, "train_instances": 24, "test_instances": 12, "all_valid": true}


Artefacts frais écrits dans data/app28-learning-to-branch-audit


### Lecture du résultat

Les lignes ci-dessus constituent la trace du run complet : 36 instances, quatre heuristiques classiques, quatre protocoles de séparation et cinq graines. Toute ligne `all_valid: true` atteste que les solutions des deux méthodes ont passé le validateur indépendant.

## 7. Synthèse multi-seed

On agrège maintenant les cinq répétitions de chaque split. Une victoire en nœuds ne suffit pas : une politique utile doit aussi amortir son inférence.


In [7]:
import json

report = json.loads((OUTPUT_DIR / "maturation_report.json").read_text(encoding="utf-8"))
summary = pd.DataFrame(report["results"])
aggregate = summary.groupby("split").agg(
    graines=("model_seed", "count"),
    top1_min=("top1_node_accuracy", "min"),
    top1_max=("top1_node_accuracy", "max"),
    ratio_noeuds_med_min=("median_node_ratio", "min"),
    ratio_noeuds_med_max=("median_node_ratio", "max"),
    ratio_temps_med_min=("median_time_ratio", "min"),
    ratio_temps_med_max=("median_time_ratio", "max"),
    victoires_temps=("strict_time_wins", "sum"),
    solutions_valides=("all_valid", "all"),
).round(3)
aggregate

,graines,top1_min,top1_max,ratio_noeuds_med_min,ratio_noeuds_med_max,ratio_temps_med_min,ratio_temps_med_max,victoires_temps,solutions_valides
split,,,,,,,,,
grouped_instance_split,5,0.580,0.580,1.000,1.000,12.183,14.990,0,True
leave_coloring_out,5,0.419,0.419,1.000,1.000,14.474,21.514,0,True
leave_latin_out,5,0.546,0.546,1.000,1.000,9.056,11.538,0,True
leave_queens_out,5,0.621,0.654,1.217,2.817,4.446,8.339,0,True


### Interprétation

Le split groupé et deux familles tenues à l'écart montrent surtout une égalité en nœuds avec MRV, tandis que le coût d'inférence domine. Le transfert vers N-reines est plus sévère : selon la graine, la politique apprise explore nettement plus de nœuds que `dom/wdeg`. Aucune répétition ne gagne en temps sur ce banc.

Ce résultat négatif est le message pédagogique central : **imiter correctement une décision locale ne prouve ni une réduction de l'arbre, ni un gain de temps, ni un transfert cross-family**.

## 8. Où passe le temps ?


In [8]:
evaluation = pd.read_csv(OUTPUT_DIR / "evaluation_runs.csv")
learned = evaluation[evaluation["heuristic"] == "learned_policy"].copy()
inference_by_split = learned.groupby("split").agg(
    part_inference_mediane=("inference_seconds", lambda s: float((s / learned.loc[s.index, "seconds"]).median())),
    temps_total_median=("seconds", "median"),
    noeuds_medians=("nodes", "median"),
).round(3)
inference_by_split

,part_inference_mediane,temps_total_median,noeuds_medians
split,,,
grouped_instance_split,0.854,0.064,28.5
leave_coloring_out,0.911,0.038,21.5
leave_latin_out,0.851,0.061,30.5
leave_queens_out,0.647,0.038,22.0


### Interprétation

La part d'inférence explique pourquoi une politique qui égale parfois MRV en nœuds peut rester beaucoup plus lente. Les temps absolus de la table dépendent de la machine ; la décomposition du même run montre néanmoins que scorer toutes les variables à chaque nœud est le poste dominant.

## 9. Trois exercices

Les exercices restent volontairement stubbés afin que le notebook s'exécute de bout en bout.

### Exercice 1 — Détecter une fuite par nœud

Construisez deux ensembles d'identifiants de nœuds et vérifiez qu'ils sont disjoints avant d'entraîner.


In [9]:
# TODO étudiant : construire train_node_ids et test_node_ids à partir de deux listes d'instances.
train_node_ids = set()
test_node_ids = set()
leakage_detected = None  # TODO étudiant : remplacer par bool(train_node_ids & test_node_ids)
print("Exercice 1 à compléter : détecter l'intersection des nœuds")

Exercice 1 à compléter : détecter l'intersection des nœuds


### Exercice 2 — Choisir une baseline sans fuite

À partir de `baseline_runs.csv`, choisissez l'heuristique de plus faible médiane sur les seules instances d'entraînement d'un split.


In [10]:
# TODO étudiant : filtrer sur les instances train avant tout groupby.
selected_baseline = None
print("Exercice 2 à compléter : sélectionner la baseline sur le train uniquement")

Exercice 2 à compléter : sélectionner la baseline sur le train uniquement


### Exercice 3 — Concevoir une règle d'abstention

Proposez une politique hybride : employer le modèle seulement lorsque son score dépasse un seuil et revenir à `dom/wdeg` sinon. Mesurez nœuds **et** temps.


In [11]:
def abstaining_policy(candidates, rows, threshold=0.9):
    # TODO étudiant : scorer, tester la confiance, puis retourner un candidat ou None.
    return None

print("Exercice 3 à compléter : ajouter une règle d'abstention")

Exercice 3 à compléter : ajouter une règle d'abstention


## 10. Limites scientifiques et suite

- Le banc contient de petits CSP binaires synthétiques et cherche une première solution ; il ne couvre ni contraintes globales natives, ni optimisation, ni solveurs industriels.
- `dom/wdeg` et activity sont des versions pédagogiques, pas des implémentations canoniques complètes.
- Les temps absolus sont machine-dépendants ; il faut conserver les ratios, les nœuds et la part d'inférence du même run.
- L'oracle imité est une heuristique, pas la meilleure décision contrefactuelle à chaque nœud.
- Une suite crédible testerait ranking groupé, distillation vers une règle légère, abstention, batching, familles plus difficiles et comparaison à budget égal.

## Conclusion

Le geste G4 devient un audit reproductible : **grouper avant d'évaluer, tenir une famille à l'écart, intégrer la politique dans le solveur et compter son coût**. Le ML ne gagne pas ici ; cette absence de gain est plus informative qu'une accuracy élevée obtenue avec fuite.

### Références

- Boussemart, F., Hemery, F., Lecoutre, C. & Sais, L. (2004). *Boosting systematic search by weighting constraints*. ECAI.
- Kotthoff, L. (2014). *Algorithm Selection for Combinatorial Search Problems: A Survey*. AI Magazine.
- Bengio, Y., Lodi, A. & Prouvost, A. (2021). *Machine Learning for Combinatorial Optimization: a Methodological Tour d'Horizon*. EJOR.
- Balcan, M.-F. (2020). *Data-Driven Algorithm Design*. Beyond the Worst-Case Analysis of Algorithms.
